[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/epipolar/epipolar.ipynb)

# Epipolar Geometry and Two-View Reconstruction

This example recovers the relative pose of two cameras and the 3D positions of observed landmarks from corresponding sight rays. Two cameras photograph the same scene from different viewpoints. A single photograph fixes only the direction toward each landmark, leaving its depth undetermined. Requiring the two sight rays toward each landmark to intersect in space determines how the second camera is turned and the direction of its baseline, and then locates every landmark up to a single overall scale.

Each sight ray is modelled as a line, the join of a camera's optical centre with an image point, and two sight rays intersect when their regressive product `rays_1 & (motor >> rays_2)` vanishes. Because a twist moves a line by their commutator, differentiating that scalar residual with respect to the camera motion gives a linear form on twists `rays_1.regressive(Twist.commutator(motor >> rays_2))`, and Gauss–Newton steps solve against the sum of squared forms measured in the second camera's kinetic energy so the unobservable scale mode along the baseline drops out cleanly. Once the pose is known, a point's squared distance to a sight ray is the quadric `(sight_rays & Point).scalar_norm_squared()`, and each triangulated landmark sits at the vertex of its two rays' summed distance quadrics.

In the matrix notation of computer vision, as laid out in [Hata and Savarese, 2021](https://web.stanford.edu/class/cs231a/course_notes/03-epipolar-geometry.pdf), coplanarity of corresponding rays reads as the essential-matrix constraint $\mathbf r_2^{\mathsf T} E\,\mathbf r_1 = 0$ with $E = [\mathbf t]_\times R$, solved by stacking outer-product rows into a $9$-column design matrix for singular value decomposition, projecting onto the equal-singular-value manifold $\operatorname{diag}(1, 1, 0)$, testing cheirality across four candidate $(R, \mathbf t)$ factorizations, and solving $2 \times 2$ normal equations per landmark.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import show_animation
from examples.estimation.epipolar import render, scenarios

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Line = ga.gatype.antibivector()                                               # Antibivector-
Motor = ga.gatype.rotor()                                                     # Rotor+
Twist = ga.gatype.bivector()                                                  # Bivector+
# Six unit masses at unit distance along the three coordinate axes of camera 2 define the kinetic-energy metric on twists.
body_masses = mv.point([[1, 0, 0], [0, 1, 0], [0, 0, 1], [-1, 0, 0], [0, -1, 0], [0, 0, -1]])  # [6] Point-
kinetic_energy = Twist.regressive(body_masses.regressive(body_masses.commutator(Twist)).sum(axis=0))  # [] Scalar+ <- (Twist+, Twist+)
point_weight = mv.w & Point                                                   # [] Scalar+ <- Point-

## Two camera views and sight rays

Camera 1 sits at the origin looking forward along the viewing axis onto an image screen one unit away, so each landmark projects to the meet `(origin & Point) ^ screen` of its sight ray with the screen. Camera 2 shares the same internal geometry at a rotated and translated pose, imaging the landmarks pulled into its own frame, and Gaussian sensor displacements perturb the image points on both screens before each optical centre joins its measured image points into calibrated sight rays.

In [ ]:
# Camera 2's true rotation and baseline translation, screen noise (~1.5 pixels on a 1000-pixel sensor), and seed.
turn_angle = scenarios.turn_angle
shift_vector = scenarios.shift_vector
sensor_noise, noise_seed = scenarios.sensor_noise, scenarios.noise_seed

In [ ]:
landmarks = scenarios.house_landmarks()                                       # [22] Point-
n_landmarks = landmarks.shape[0]
# Camera 1 sits at the world origin looking along +z onto the image plane z == 1.
origin = mv.w.dual()                                                          # [] Point-
screen = mv.z - mv.w                                                          # [] Plane+
camera = (origin & Point) ^ screen                                            # [] Point- <- Point-

shift_generator = (stack((mv.xw, mv.yw, mv.zw)) * shift_vector).sum(axis=0) / 2  # [] Twist+
shift_motor = shift_generator.exp()                                           # [] Motor+
turn_motor = (mv.zx * (turn_angle / 2)).exp()                                 # [] Motor+
true_motor = (shift_motor * turn_motor).normalized()                          # [] Motor+

# Dividing each projected point by its projective weight places all image points at unit weight on z == 1.
raw_images = camera(stack([landmarks, true_motor << landmarks]))              # [2, n_landmarks] Point-
images = raw_images / (mv.w & raw_images)                                     # [2, n_landmarks] Point-
screen_axes = mv.direction([[1, 0, 0], [0, 1, 0]])                            # [2] Point-
offsets = np.random.default_rng(noise_seed).normal(0.0, sensor_noise, size=images.shape + (2,))
measured_images = images + (screen_axes * offsets).sum(axis=-1)               # [2, n_landmarks] Point-
rays = (origin & measured_images).normalized()                                # [2, n_landmarks] Line-

In [ ]:
print("pinhole camera map from world points to screen points:\n", camera.coefficient_table())

## Relative pose and landmark triangulation

Starting from the unrotated baseline shift, each Gauss–Newton step solves for the twist that drives the ray-crossing residuals toward zero while truncating the zero-curvature scale mode along the baseline in camera 2's kinetic-energy metric. At any candidate pose, summing the squared-distance quadrics of paired sight rays and pinning homogeneous weight with `point_weight * point_weight` triangulates every landmark in a single linear solve.

In [ ]:
n_iterations = scenarios.n_iterations

In [ ]:
def pose_step(rays_1: Line, rays_2: Line, motor: Motor) -> Motor:
    """One Gauss–Newton step from `motor` (`[] Motor+`) in camera 1's frame moves camera 2 toward intersecting sight rays (`[...] Line-`) while dropping the unobservable baseline-scale mode."""
    # Two lines intersect (are coplanar) when their regressive product vanishes, giving one scalar residual per ray pair.
    residuals = rays_1 & (motor >> rays_2)                                    # [...] Scalar-
    # A twist moves a line by their commutator, so differentiating the crossing residual yields a linear form on twists.
    jacobian = rays_1.regressive(Twist.commutator(motor >> rays_2))           # [...] Scalar- <- Twist+
    # Scalar residuals square directly without a metric, forming the Gauss-Newton curvature and descent forms over twists.
    curvature = (jacobian * jacobian).sum(axis=0)                             # [] Scalar+ <- (Twist+, Twist+)
    descent = -(residuals * jacobian).sum(axis=0)                             # [] Scalar+ <- Twist+
    # Measuring twists in camera 2's kinetic energy isolates the zero-curvature scale mode along the baseline for rcond truncation.
    step_metric = kinetic_energy(motor << Twist, motor << Twist)              # [] Scalar+ <- (Twist+, Twist+)
    step_twist = curvature.lstsq(descent, step_metric, rcond=1e-4)            # [] Twist+
    step_motor = (step_twist * 0.5).exp()                                     # [] Motor+
    return (step_motor * motor).normalized()                                  # [] Motor+


def triangulated_points(rays_1: Line, rays_2: Line, motor: Motor) -> Point:
    """Summing the squared-distance quadrics of `rays_1` and `motor >> rays_2` (`[...] Line-`) locates each triangulated landmark (`[...] Point-`) in camera 1's frame."""
    # Joining a ray with a point spans the plane through both whose squared norm equals the point's squared distance to the ray.
    sight_rays = stack([rays_1, motor >> rays_2])                             # [2, ...] Line-
    distance_quadrics = (sight_rays & Point).scalar_norm_squared().sum(axis=0)  # [...] Scalar+ <- (Point-, Point-)
    # Adding the projective weight dyad pins homogeneous scale without shifting the quadric vertex.
    return (distance_quadrics + point_weight * point_weight).solve(point_weight).normalized()  # [...] Point-

In [ ]:
def reconstruction_states(rays_1: Line, rays_2: Line, motor: Motor) -> Iterator[tuple[Motor, Point]]:
    """Yields camera 2's relative pose (`[] Motor+`) and the triangulated landmarks (`[...] Point-`) across successive Gauss–Newton steps."""
    while True:
        yield motor, triangulated_points(rays_1, rays_2, motor)
        motor = pose_step(rays_1, rays_2, motor)


def pose_and_landmark_errors(
    landmarks: Point,
    reconstructed_landmarks: Point,
    true_motor: Motor,
    estimated_motor: Motor,
) -> tuple[Scalar, Scalar, Scalar]:
    """Axis-plane cosines (`[3] Scalar+`), baseline-direction cosine (`[] Scalar+`), and baseline-scaled landmark RMS error (`[] Scalar+`) compare `estimated_motor` and `reconstructed_landmarks` (`[...] Point-`) against ground truth."""
    axes = stack((mv.x, mv.y, mv.z))                                          # [3] Plane+
    axis_cosines = (estimated_motor >> axes).normalized() | (true_motor >> axes).normalized()  # [3] Scalar+
    # Subtracting two unit-weight points yields the ideal displacement direction along each camera baseline.
    true_baseline = (true_motor >> origin) - origin                           # [] Point-
    estimated_baseline = (estimated_motor >> origin) - origin                 # [] Point-
    baseline_cosine = true_baseline.dot(estimated_baseline) / (true_baseline.antinorm() * estimated_baseline.antinorm())  # [] Scalar+
    scale = true_baseline.antinorm() / estimated_baseline.antinorm()          # [] Scalar+
    unit_landmarks = reconstructed_landmarks / (mv.w & reconstructed_landmarks)  # [...] Point-
    displacements = (unit_landmarks * scale + origin * (1.0 - scale)) - landmarks  # [...] Point-
    landmark_rms = displacements.dot(displacements).mean(axis=0).square_root()  # [] Scalar+
    return axis_cosines, baseline_cosine, landmark_rms


initial_motor = shift_motor.normalized()                                      # [] Motor+
iterates = islice(reconstruction_states(rays[0], rays[1], initial_motor), n_iterations + 1)
estimated_motors, reconstructed_landmarks = (stack(path) for path in zip(*iterates))  # [n_iterations + 1] Motor+, [n_iterations + 1, n_landmarks] Point-
estimated_motor = estimated_motors[-1]                                        # [] Motor+
axis_cosines, baseline_cosine, landmark_rms = pose_and_landmark_errors(
    landmarks, reconstructed_landmarks[-1], true_motor, estimated_motor,
)

In [ ]:
print("axis-plane turn errors (deg):", np.degrees(np.arccos(np.minimum(axis_cosines.to_array(), 1))))
print("baseline direction error (deg):", np.degrees(np.arccos(np.minimum(baseline_cosine.to_array(), 1))))
print("landmark RMS error at true baseline scale:", landmark_rms.to_array())

## Reconstruction trajectory

Interpolating along the logarithm screw between consecutive Gauss–Newton motors shows how the triangulated landmarks evolve as the rays come into alignment, scaled about camera 1 to the true baseline length. This cell takes about five seconds to ray-trace the animation.

In [ ]:
n_easing, n_hold, frame_duration_ms = scenarios.n_easing, scenarios.n_hold, scenarios.frame_duration_ms

In [ ]:
show_animation(
    render.reconstruction_animation(
        landmarks,
        scenarios.house_edges,
        true_motor,
        scenarios.eased_states(rays, estimated_motors, n_easing, n_hold),
    ),
    "epipolar_reconstruction",
    frame_duration_ms,
);

Camera 1 is drawn in blue, camera 2 is grey at its true pose and green at its estimated pose, the reference wireframe house and landmarks are grey, and the triangulated landmarks and sampled sight rays are coloured by landmark index. From the unrotated starting guess the skew rays miss one another and scatter the triangulated points, before three Gauss–Newton updates align camera 2 and pull the landmarks onto the house within the sensor noise.

In [ ]:
# checks
# Noise-free sight rays intersect exactly at the true pose and triangulate the landmarks to machine precision.
clean_rays = (origin & images).normalized()                                   # [2, n_landmarks] Line-
assert ((clean_rays[0] & (true_motor >> clean_rays[1])).abs() < 1e-12).all()
clean_triangulated = triangulated_points(clean_rays[0], clean_rays[1], true_motor)  # [n_landmarks] Point-
assert ((clean_triangulated - landmarks).antinorm() < 1e-10).all()
# Gauss-Newton updates reduce the noisy ray-crossing residual and recover the pose and house within sensor noise.
initial_residual = (rays[0] & (initial_motor >> rays[1])).squared().mean(axis=0).square_root()  # [] Scalar+
final_residual = (rays[0] & (estimated_motor >> rays[1])).squared().mean(axis=0).square_root()  # [] Scalar+
assert (final_residual < 0.05 * initial_residual).all()
assert (axis_cosines > np.cos(np.radians(0.5))).all()
assert (baseline_cosine > np.cos(np.radians(2.0))).all()
assert (landmark_rms < 0.1).all()